# RARS-v2 boundary-loss feasibility (Colab T4)

Development-only experiment using the deterministic BEIR NQ **train** split. The closed NQ test queries, test qrels, Stage-3 evaluation arrays, and post-hoc per-query outputs are never read. Candidate residual bundles live on Colab's local disk; only trained outputs are persisted to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json, os, re, shutil, subprocess, sys
from pathlib import Path

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'faiss-gpu-cu12==1.12.0', 'pytest>=8,<9'], check=True)

SOURCE_ARTIFACT_ROOT = Path('/content/drive/MyDrive/rars-beir-nq-confirmation-v2')
LOCAL_WORK_ROOT = Path('/content/rars-v2-boundary-work')
BUNDLE_ROOT = LOCAL_WORK_ROOT / 'bundles'
OUTPUT_ROOT = Path('/content/drive/MyDrive/rars-v2-boundary-loss-feasibility-v1')
REPO = Path('/content/Embedding_Compression_for_RAG_Retrieval_rars_v2')
BRANCH = 'experiment/rars-v2-boundary-consolidation'
CORE_COMMIT = '530e7c3a7933facc23b51e578b9718b3050f43ba'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Source:', SOURCE_ARTIFACT_ROOT)
print('Local bundles:', BUNDLE_ROOT)
print('Persistent outputs:', OUTPUT_ROOT)

In [ ]:
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'switch', '--detach', 'FETCH_HEAD'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch',
                    'https://github.com/ravan-chuang/Embedding_Compression_for_RAG_Retrieval.git',
                    str(REPO)], check=True)
head = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
status = subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True)
ancestor = subprocess.run(['git', '-C', str(REPO), 'merge-base', '--is-ancestor',
                           CORE_COMMIT, head]).returncode == 0
assert ancestor, (CORE_COMMIT, head)
assert not status, status
assert SOURCE_ARTIFACT_ROOT.is_dir(), SOURCE_ARTIFACT_ROOT
local_free_gb = shutil.disk_usage('/content').free / 1e9
assert local_free_gb >= 6, f'Need at least 6 GB local disk; found {local_free_gb:.1f} GB'
print('Clean experiment checkout:', head)
print(f'Local free disk: {local_free_gb:.1f} GB')

In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-q',
                'tests/test_build_rars_v2_boundary_bundles.py',
                'tests/test_boundary_loss_sidecar.py',
                'tests/test_boundary_aware_sidecar.py'],
               cwd=REPO, check=True)

## Build compact development bundles

This reads only Stage-1/Stage-2 artifacts and `qrels/train.tsv`. Candidate-union residuals are reconstructed into `/content`, so a runtime disconnect requires rebuilding this cell but does not damage the frozen source artifacts.

In [ ]:
subprocess.run([
    sys.executable, str(REPO / 'scripts/build_rars_v2_boundary_bundles.py'),
    '--artifact-root', str(SOURCE_ARTIFACT_ROOT),
    '--output-root', str(BUNDLE_ROOT),
    '--residual-batch-size', '20000',
], check=True)
summary = json.loads((BUNDLE_ROOT / 'bundle_build_summary.json').read_text())
assert summary['test_qrels_accessed'] is False
assert summary['nq_test_retuning_authorized'] is False
print(json.dumps(summary, indent=2))
subprocess.run(['du', '-sh', str(BUNDLE_ROOT)], check=True)

## Seed-42 smoke test (1 epoch)

This checks end-to-end training, fixed-scale QAT, and validation scoring. It is not a reportable result and does not trigger the go/no-go decision.

In [ ]:
SMOKE_OUTPUT = OUTPUT_ROOT / 'seed42-smoke-1epoch'
subprocess.run([
    sys.executable, str(REPO / 'scripts/train_boundary_loss_sidecar.py'),
    '--bundle-dir', str(BUNDLE_ROOT / 'train'),
    '--validation-bundle-dir', str(BUNDLE_ROOT / 'validation'),
    '--output-dir', str(SMOKE_OUTPUT),
    '--rank', '16', '--top-b', '40', '--final-k', '10',
    '--epochs', '1', '--batch-size', '2048', '--seed', '42',
    '--skip-full-encoding', '--device', 'cuda',
], check=True)
smoke = json.loads((SMOKE_OUTPUT / 'training_summary.json').read_text())
assert smoke['test_qrels_accessed'] is False
assert smoke['final_loss'] == smoke['final_loss']
print(json.dumps(smoke, indent=2))

## Registered seed-42 feasibility run (5 epochs)

Run this only after the smoke cell completes successfully. Hyperparameters below are the protocol defaults.

In [ ]:
RUN_OUTPUT = OUTPUT_ROOT / 'seed42-5epochs'
subprocess.run([
    sys.executable, str(REPO / 'scripts/train_boundary_loss_sidecar.py'),
    '--bundle-dir', str(BUNDLE_ROOT / 'train'),
    '--validation-bundle-dir', str(BUNDLE_ROOT / 'validation'),
    '--output-dir', str(RUN_OUTPUT),
    '--rank', '16', '--top-b', '40', '--final-k', '10',
    '--negative-window', '10', '--max-negatives-per-positive', '8',
    '--margin', '0.05', '--epochs', '5', '--batch-size', '2048',
    '--learning-rate', '0.001', '--weight-decay', '0.0001',
    '--correction-l2', '0.0001', '--seed', '42',
    '--skip-full-encoding', '--device', 'cuda',
], check=True)
result = json.loads((RUN_OUTPUT / 'training_summary.json').read_text())
print(json.dumps(result, indent=2))

In [ ]:
metrics = result['validation']
retained = metrics['int8_fraction_of_fp32_gain']
go_checks = {
    'gain_over_base_at_least_0.01': metrics['int8_gain_over_base'] >= 0.01,
    'beats_storage_matched_pca': metrics.get('beats_storage_matched_pca', False),
    'retains_70pct_fp32_gain': retained is not None and retained >= 0.70,
    'improved_exceeds_harmed': metrics['improved_queries'] > metrics['harmed_queries'],
}
decision = {
    'status': 'GO' if all(go_checks.values()) else 'NO_GO_OR_REVISE',
    'checks': go_checks,
    'validation': metrics,
    'test_qrels_accessed': False,
}
(RUN_OUTPUT / 'go_no_go.json').write_text(json.dumps(decision, indent=2) + '\n')
print(json.dumps(decision, indent=2))